In [1]:
# Cell 1: Setup
import duckdb
import os
import pandas as pd
import subprocess
import json

db_path = os.path.expanduser('~/trading_infra/storage/trading.duckdb')
conn = duckdb.connect(db_path)

print("="*70)
print("HOLDINGS SCANNER - Sector Deep Dive")
print("="*70)


IOException: IO Error: Could not set lock on file "/home/trading/trading_infra/storage/trading.duckdb": Conflicting lock is held in /usr/bin/python3.13 (PID 1275678) by user sysadmin. See also https://duckdb.org/docs/stable/connect/concurrency

In [ ]:
# Cell 2: Select Sector to Analyze
# CHANGE THIS TO YOUR TARGET SECTOR
SECTOR_ETF = 'SPY'  # Options: SPY, QQQ, IWM, DIA, GLD, SLV, VXX, XLP, XLB, XLV, XLRE, XLK, XLF, XLI, XLY, XLC, XLU, XME

print(f"\n📊 Analyzing holdings of {SECTOR_ETF}\n")


In [ ]:
# Cell 3: Load Top Holdings
# Note: yfinance doesn't provide detailed holdings, so we'll work with available data
# In production, you'd pull from proprietary sources or fund websites

print(f"Top holdings of {SECTOR_ETF}:")
print("(Holdings data would be pulled from financial data providers)")
print("\nTo add holdings data:")
print("1. Manually add to etf_holdings table:")
print("   INSERT INTO etf_holdings VALUES ('XLK', 'MSFT', 0.22, NOW())")
print("2. Or import from CSV")


In [ ]:
# Cell 4: Check for Volume Anomalies (Mock Data for Now)
# In production, this queries real holdings you've ingested

print("\n" + "="*70)
print("VOLUME ANOMALIES IN SECTOR HOLDINGS")
print("="*70)

# For now, show general sector anomalies
anomalies = conn.execute(f"""
    SELECT 
        ticker,
        volume_ratio,
        signal_strength,
        timestamp
    FROM volume_anomalies
    WHERE signal_strength IN ('strong', 'moderate')
    ORDER BY volume_ratio DESC
    LIMIT 10
""").fetch_df()

if not anomalies.empty:
    print(f"\nFound {len(anomalies)} anomalies:")
    print(anomalies.to_string(index=False))
else:
    print("\nNo recent anomalies detected")


In [ ]:
# Cell 5: Simulate Holdings Scan
# This is what will happen once you add holdings to the database

print("\n" + "="*70)
print("MOCK HOLDINGS ANALYSIS FOR XLK")
print("="*70)

mock_holdings = pd.DataFrame({
    'Ticker': ['MSFT', 'AAPL', 'NVDA', 'GOOGL', 'META'],
    'Weight_%': [22.5, 15.3, 12.8, 11.2, 8.9],
    'Current_Price': [435.62, 232.45, 875.32, 168.53, 510.87],
    'Call_Volume_Ratio': [2.3, 1.8, 3.1, 1.5, 2.8],
    'IV_Rank': [72, 68, 78, 65, 71]
})

print("\n" + mock_holdings.to_string(index=False))

print("\n🎯 HIGH-PROBABILITY SETUPS (Call Volume Ratio > 2.0):")
high_prob = mock_holdings[mock_holdings['Call_Volume_Ratio'] > 2.0]
for idx, row in high_prob.iterrows():
    print(f"  • {row['Ticker']}: {row['Call_Volume_Ratio']}x call volume, IV rank {row['IV_Rank']}")


In [ ]:
# Cell 6: LLM Analysis of Holdings
print("\n" + "="*70)
print("PLUTUS: HOLDINGS RECOMMENDATION")
print("="*70)

prompt = f"""You are analyzing {SECTOR_ETF} sector holdings showing high call volume.

Top candidates:
{high_prob[['Ticker', 'Weight_%', 'Call_Volume_Ratio', 'IV_Rank']].to_string(index=False)}

For each ticker showing >2.0x call volume:
1. What accumulation pattern is this suggesting?
2. Recommend bull call spread strikes
3. Target risk/reward ratio
4. Confidence level (1-10)

Keep it concise and trade-ready."""

print("Querying Plutus...")

result = subprocess.run(
    ['ollama', 'run', '0xroyce/plutus'],
    input=prompt.encode(),
    capture_output=True,
    timeout=120
)

analysis = result.stdout.decode('utf-8', errors='ignore')
print("\n" + analysis)


In [ ]:
# Cell 7: Action Items
print("\n" + "="*70)
print("NEXT STEPS")
print("="*70)
print("""
✓ If high call volume + rising IV confirmed:
  1. Open TradingView
  2. Pull up daily + 15m chart for candidate
  3. Identify support level for entry
  4. Set up bull call spread entry
  5. Execute on Webull

✓ Log trade in execution/trade_log.md
✓ Track P&L

To use real holdings data:
  1. Download from fund website or financial data provider
  2. INSERT into etf_holdings table
  3. Re-run this notebook
""")

conn.close()
print("\n✓ Holdings scan complete")
